# RPA: Extração de clientes e envio ao N8N

Este notebook faz o papel do "robô": abre a página de clientes (GitHub Pages), lê a tabela
e envia os dados para o Webhook do N8N, que gera e envia as mensagens personalizadas.

**Antes de rodar:**
1. Troque `SEU-USUARIO` em `URL_PAGINA`.
2. Guarde a URL do Webhook em *Secrets* do Colab (ícone de chave), com o nome `N8N_WEBHOOK`.
   Não escreva a URL direto no código nem publique no GitHub.

# 1. Dependências

In [ ]:
!pip install requests beautifulsoup4

# 2. Configuração

In [ ]:
import os

# Página de clientes publicada no GitHub Pages (pasta docs/)
URL_PAGINA = "https://SEU-USUARIO.github.io/dio-lab-assistente-investimentos-rpa-n8n/"


def obter_webhook():
    """Lê a URL do Webhook de um Secret do Colab ou da variável de ambiente N8N_WEBHOOK."""
    try:
        from google.colab import userdata
        return userdata.get("N8N_WEBHOOK")
    except Exception:
        return os.getenv("N8N_WEBHOOK")


N8N_WEBHOOK = obter_webhook()

# 3. Extração (RPA)

In [ ]:
import requests
from bs4 import BeautifulSoup

# 1) Baixar a página
resposta = requests.get(URL_PAGINA, timeout=30)
resposta.raise_for_status()
resposta.encoding = "utf-8"

# 2) Ler a tabela de clientes
soup = BeautifulSoup(resposta.text, "html.parser")
linhas = soup.select("#clientes tbody tr")

clientes = []
for linha in linhas:
    colunas = linha.find_all("td")
    clientes.append({
        "nome":   colunas[0].get_text(strip=True),
        "email":  colunas[1].get_text(strip=True),
        "saldo":  colunas[2].get_text(strip=True),
        "perfil": colunas[3].get_text(strip=True),
    })

# 3) Mostrar o resultado
print(f"Clientes encontrados: {len(clientes)}")
for cliente in clientes:
    print(cliente)

# 4. Envio ao N8N (Webhook)

- Para testar, use a **Test URL** (`.../webhook-test/clientes`) com o N8N em *Listen for test event*.
- Depois de ativar o workflow, use a **Production URL** (`.../webhook/clientes`).
- O N8N só responde depois de gerar todas as mensagens, por isso o `timeout` é alto.

In [ ]:
if not clientes:
    raise SystemExit("Nenhum cliente encontrado: confira a URL_PAGINA e o seletor.")

if not N8N_WEBHOOK:
    raise SystemExit("Defina o Secret N8N_WEBHOOK (veja as instruções no topo do notebook).")

resp = requests.post(N8N_WEBHOOK, json={"clientes": clientes}, timeout=120)
print("Status:", resp.status_code)
print("Resposta:", resp.text)